# СИЛАНТ: сравнение с предобученной моделью Chronos-2

## Цель

 Chronos-2 (Amazon, октябрь 2025) — предобученная модель временных рядов: она обучена на большом массиве чужих рядов и прогнозирует без подстройки под наши 13 кварталов, поэтому не переобучается. Умеет учитывать внешние факторы с известными будущими значениями. Здесь она проверяется **точно так же, как модели отчёта**: те же 5 точек начала прогноза (IV кв. 2024 — IV кв. 2025), горизонт 1–4 квартала, та же метрика (средняя |ln(факт/прогноз)|), те же 13 рядов.

Два варианта: без факторов и с ключевой ставкой с лагом 2 квартала (как в ARX). **Важно:** вариант «Chronos-2 + ставка» на проверке получает фактическую будущую ставку (oracle), поэтому его результат оптимистичен и на равных с честной проверкой его сравнивать нельзя. В пайплайне ARX до 15.10.2026 тоже получал фактическую ставку; теперь в его проверке ставка после начала прогноза замораживается на последнем известном значении (см. `src/silant_forecast.py`, `rolling_origin`).

**Как запустить.** Откройте ноутбук из папки `chronos2/` репозитория и выполните все ячейки (Run All). Первый запуск ставит Chronos-2 и PyTorch (около 1 ГБ, нужен интернет) и скачивает веса модели с Hugging Face; на CPU расчёт занимает 5–10 минут. Результаты пишутся в `chronos2/results/`. Модель детерминирована (квантили, без случайной выборки). Выводы ячеек в файле сохранены с оригинального запуска в облачном ноутбуке (CPU).

*How to run (EN): open this notebook from the `chronos2/` folder and Run All. It installs `chronos-forecasting`, downloads `amazon/chronos-2` from Hugging Face and writes results to `chronos2/results/`. Cell outputs are kept from the original cloud-notebook run (CPU).*


## 1. Код и таблицы пайплайна

In [1]:
import os, sys, subprocess
from pathlib import Path

NB_DIR = Path.cwd()                                # run the notebook from the chronos2/ folder
ROOT = NB_DIR.parent if NB_DIR.name == 'chronos2' else NB_DIR
os.chdir(ROOT)                                     # repo root: data/processed/ must exist
need = ['market_long.csv', 'macro_quarterly_wide.csv', 'backtest_rolling_origin.csv']
if not all(os.path.exists('data/processed/' + f) for f in need):
    print('Pipeline tables are missing, running update.py (about 2 minutes)...')
    subprocess.run([sys.executable, 'update.py'], check=True)
OUT_DIR = ROOT / 'chronos2' / 'results'
print('Repo root:', os.getcwd(), '| results ->', OUT_DIR)

(output of the original run was cleared because it contained local paths; rerun the cell to see it)


## 2. Установить Chronos-2
Ячейка ставит `chronos-forecasting` через `%pip` и скачивает веса `amazon/chronos-2`. Строка `%pip` предназначена для запуска в Colab/GPU или в любом окружении, где пакета ещё нет; локально можно поставить его заранее (`pip install chronos-forecasting`).

In [2]:
# run in Colab/GPU (or any environment without chronos-forecasting); skip if it is already installed
%pip install -q "chronos-forecasting>=2.0" pandas
import torch
try:
    from chronos import Chronos2Pipeline as _P
except ImportError:
    from chronos import BaseChronosPipeline as _P
device = 'cuda' if torch.cuda.is_available() else 'cpu'
pipe = _P.from_pretrained('amazon/chronos-2', device_map=device)
predict = pipe.predict_df
import chronos
print('chronos', getattr(chronos, '__version__', '?'), '| устройство:', device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 3.7 MB/s eta 0:00:00


config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  478MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

chronos 2.3.2 | устройство: cpu


## 3. Подготовка данных и функции оценки
Ряды собираются из `market_long.csv` так же, как в пайплайне (проверка ниже: факты совпадают с файлом проверки на истории).

In [3]:
# ==== ядро сравнения (вставляется в блокнот) ====
import numpy as np, pandas as pd

P = 'data/processed/'
ORIGINS = ['2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4']   # те же 5 точек, что в отчёте
H = 4
QL = [0.025, 0.1, 0.5, 0.9, 0.975]
LAG = 2

ml = pd.read_csv(P + 'market_long.csv', encoding='utf-8-sig')
ml['quarter'] = pd.PeriodIndex(ml['quarter'], freq='Q')
BIG = ['1–1,8 т', '2–2,5 т', '3–3,5 т', '3,8–5 т']

def build_nodes(ml):
    S = {}
    for t in ['ДВС', 'Электро']:
        d = ml[ml.type == t]
        S[t] = d.groupby('quarter').units.sum()
        for s in BIG:
            S[f'{t} {s}'] = d[d.segment == s].groupby('quarter').units.sum()
        S[f'{t} 5,5 т+'] = d[d.cap_min >= 5.5].groupby('quarter').units.sum()
    S['Рынок'] = S['ДВС'] + S['Электро']
    return pd.DataFrame(S).sort_index().astype(float)

S = build_nodes(ml)
NODES = list(S.columns)
SEG8 = [f'{t} {s}' for t in ['ДВС', 'Электро'] for s in BIG]

mw = pd.read_csv(P + 'macro_quarterly_wide.csv', encoding='utf-8-sig')
mw['quarter'] = pd.PeriodIndex(mw['quarter'], freq='Q')
KR = mw.set_index('quarter')['key_rate_avg']
KR2 = pd.Series(KR.values, index=KR.index + LAG)        # ставка с лагом 2 квартала, как в ARX

def ts(q):  # квартал -> дата начала квартала
    return q.to_timestamp(how='start')

def make_frames(origin, with_kr):
    o = pd.Period(origin, freq='Q')
    hist = S.loc[:o]
    fut_q = pd.period_range(o + 1, o + H, freq='Q')
    ctx, fut = [], []
    for n in NODES:
        for q, v in hist[n].items():
            r = {'id': n, 'timestamp': ts(q), 'target': v}
            if with_kr: r['kr_lag2'] = KR2[q]
            ctx.append(r)
        if with_kr:
            for q in fut_q:
                fut.append({'id': n, 'timestamp': ts(q), 'kr_lag2': KR2[q]})
    return pd.DataFrame(ctx), (pd.DataFrame(fut) if with_kr else None), fut_q

def qcol(df, q):
    for c in (str(q), q, f'{q:g}'):
        if c in df.columns: return c
    raise KeyError(f'нет столбца квантиля {q}: {list(df.columns)}')

def run_chronos(predict, origin, with_kr):
    ctx, fut, fut_q = make_frames(origin, with_kr)
    kw = dict(prediction_length=H, quantile_levels=QL, id_column='id', timestamp_column='timestamp', target='target')
    if fut is not None: kw['future_df'] = fut
    pr = predict(ctx, **kw)
    pr = pr.copy(); pr['timestamp'] = pd.to_datetime(pr['timestamp'])
    out = []
    for _, r in pr.iterrows():
        q = pd.Period(r['timestamp'], freq='Q')
        h = fut_q.get_loc(q) + 1
        out.append({'origin': origin, 'target': str(q), 'h': h, 'node': r['id'],
                    **{f'q{x}': max(float(r[qcol(pr, x)]), 0.0) for x in QL}})
    return pd.DataFrame(out)

def offset(n): return 10.0 if '5,5' in n else 0.0

def score(fc, model):
    rows = []
    for _, r in fc.iterrows():
        q = pd.Period(r['target'], freq='Q')
        if q not in S.index: continue
        n = r['node']; act = S.loc[q, n]; pt = r['q0.5']
        rows.append({'origin': r['origin'], 'target': r['target'], 'h': r['h'], 'model': model, 'node': n,
                     'actual': act, 'point': pt,
                     'abs_log_err': abs(np.log((act + offset(n)) / (pt + offset(n)))),
                     'log_bias': np.log((pt + offset(n)) / (act + offset(n))),
                     'in80': float(r['q0.1'] <= act <= r['q0.9']), 'in95': float(r['q0.025'] <= act <= r['q0.975'])})
    # доля электро — по медианам ДВС и Электро
    w = fc.pivot_table(index=['origin', 'target', 'h'], columns='node', values='q0.5').reset_index()
    for _, r in w.iterrows():
        q = pd.Period(r['target'], freq='Q')
        if q not in S.index: continue
        act = S.loc[q, 'Электро'] / S.loc[q, 'Рынок']; pt = r['Электро'] / (r['Электро'] + r['ДВС'])
        rows.append({'origin': r['origin'], 'target': r['target'], 'h': r['h'], 'model': model, 'node': 'Доля электро',
                     'actual': act, 'point': pt, 'abs_log_err': abs(act - pt), 'log_bias': pt - act,
                     'in80': np.nan, 'in95': np.nan})
    return pd.DataFrame(rows)

def summary_table(bt):
    g = bt.groupby(['model', 'node'])
    e = g.abs_log_err.mean().unstack()
    t = pd.DataFrame({'Рынок': e['Рынок'], 'ДВС': e['ДВС'], 'Электро': e['Электро'],
                      '8 крупных сегментов': e[SEG8].mean(axis=1), 'Доля электро, п. п.': e['Доля электро'] * 100})
    b = g.log_bias.mean().unstack()
    t['Смещение рынка'] = np.exp(b['Рынок']) - 1
    cov = pd.to_numeric(bt.loc[bt.node != 'Доля электро', 'in80'], errors='coerce').groupby(bt.loc[bt.node != 'Доля электро', 'model']).mean()
    t['Покрытие 80 %-интервала'] = cov
    return t

def pipeline_scores(src='rec'):
    b = pd.read_csv(P + 'backtest_rolling_origin.csv', encoding='utf-8-sig')
    b = b[b.src == src].copy()
    b['log_bias'] = np.where(b.node == 'Доля электро', b.point - b.actual,
                             np.log((b.point + b.node.map(offset)) / (b.actual + b.node.map(offset))))
    b['in80'] = b['in80'].astype(float)
    return b

def combo_points(base_bt, ch_bt, name, models=('Наивный', 'ETS', 'ARX-ставка')):
    """Комбинация как среднее геометрическое точечных прогнозов (несогласованных) выбранных моделей [+ Chronos-2].
    Считается одинаково для варианта с Chronos-2 и без него, поэтому их можно сравнивать между собой."""
    keys = ['origin', 'target', 'node']
    b = base_bt[base_bt.node != 'Доля электро']
    m = b[b.model == models[0]][keys + ['h', 'actual']]
    cols = []
    for mod in models:
        m = m.merge(b[b.model == mod][keys + ['point']].rename(columns={'point': mod}), on=keys); cols.append(mod)
    if ch_bt is not None:
        m = m.merge(ch_bt[ch_bt.node != 'Доля электро'][keys + ['point']].rename(columns={'point': 'Chronos'}), on=keys)
        cols.append('Chronos')
    off = m.node.map(offset)
    m['point'] = np.exp(np.mean([np.log(m[c] + off) for c in cols], axis=0)) - off
    m['abs_log_err'] = np.abs(np.log((m.actual + off) / (m.point + off)))
    m['log_bias'] = np.log((m.point + off) / (m.actual + off))
    m['in80'] = np.nan; m['model'] = name
    sh = m[m.node.isin(['ДВС', 'Электро'])].pivot_table(index=['origin', 'target', 'h'], columns='node', values=['point', 'actual']).reset_index()
    sh_rows = pd.DataFrame({'origin': sh['origin'], 'target': sh['target'], 'h': sh['h'], 'node': 'Доля электро', 'model': name,
                            'actual': sh[('actual', 'Электро')] / (sh[('actual', 'Электро')] + sh[('actual', 'ДВС')]),
                            'point': sh[('point', 'Электро')] / (sh[('point', 'Электро')] + sh[('point', 'ДВС')])})
    sh_rows['abs_log_err'] = (sh_rows.actual - sh_rows.point).abs(); sh_rows['log_bias'] = sh_rows.point - sh_rows.actual
    return pd.concat([m[['origin', 'target', 'h', 'model', 'node', 'actual', 'point', 'abs_log_err', 'log_bias', 'in80']], sh_rows], ignore_index=True)

def forecast_4q(predict, with_kr, origin='2026Q1'):
    fc = run_chronos(predict, origin, with_kr)
    tot = fc[fc.node.isin(['Рынок', 'ДВС', 'Электро'])].groupby('node')[['q0.1', 'q0.5', 'q0.9']].sum()
    return fc, tot

b_ = pd.read_csv(P + 'backtest_rolling_origin.csv', encoding='utf-8-sig')
b_ = b_[b_.node != 'Доля электро']
mism = sum(abs(S.loc[pd.Period(r.target, freq='Q'), r.node] - r.actual) > 1e-6 for r in b_.itertuples())
print('Рядов:', len(NODES), '| расхождений фактов с пайплайном:', mism)

Рядов: 13 | расхождений фактов с пайплайном: 0


## 4. Проверка на истории: 5 точек × горизонт 1–4 квартала

In [4]:
res, FCS = [], {}
for with_kr, name in [(False, 'Chronos-2'), (True, 'Chronos-2 + ставка')]:
    fc = pd.concat([run_chronos(predict, o, with_kr) for o in ORIGINS], ignore_index=True)
    FCS[name] = fc
    res.append(score(fc, name))
def last_value(ctx, prediction_length, quantile_levels, id_column, timestamp_column, target, future_df=None):
    # простейшая база: повторить последнее фактическое значение (без сезонной поправки)
    out = []
    for i, g in ctx.groupby(id_column):
        last = g[target].iloc[-1]; t0 = g[timestamp_column].max()
        for h in range(1, prediction_length + 1):
            out.append({'id': i, 'timestamp': t0 + pd.DateOffset(months=3 * h), **{str(q): last for q in quantile_levels}})
    return pd.DataFrame(out)
LV = score(pd.concat([run_chronos(last_value, o, False) for o in ORIGINS], ignore_index=True), 'Последнее значение (без сезонности)')
LV['in80'] = np.nan
res.append(LV)
CH = pd.concat(res, ignore_index=True)
BASE, REC = pipeline_scores('base'), pipeline_scores('rec')
ALL = pd.concat([
    REC,
    CH,
    combo_points(BASE, None, 'Комбинация 3 (точки) — база сравнения'),
    combo_points(BASE, CH[CH.model == 'Chronos-2'], 'Комбинация 3 + Chronos-2'),
    combo_points(BASE, CH[CH.model == 'Chronos-2 + ставка'], 'Комбинация 3 + Chronos-2 со ставкой'),
], ignore_index=True)
T = summary_table(ALL)
order = ['Последнее значение (без сезонности)', 'Наивный', 'Сезонный наивный', 'ETS', 'ARX-ставка', 'Комбинация', 'Chronos-2', 'Chronos-2 + ставка',
         'Комбинация 3 (точки) — база сравнения', 'Комбинация 3 + Chronos-2', 'Комбинация 3 + Chronos-2 со ставкой']
T = T.loc[[m for m in order if m in T.index]]
fmt = T.copy()
for c in ['Рынок', 'ДВС', 'Электро', '8 крупных сегментов', 'Смещение рынка', 'Покрытие 80 %-интервала']:
    fmt[c] = (T[c] * 100).map(lambda v: '' if pd.isna(v) else f'{v:.1f} %')
fmt['Доля электро, п. п.'] = T['Доля электро, п. п.'].map(lambda v: f'{v:.1f}')
print('Средняя ошибка |ln(факт/прогноз)|, горизонт 1–4 кв. Модели отчёта — согласованные прогнозы (как в табл. 8);')
print('покрытие 80 %-интервала — без калибровки ширины. «Комбинация 3 (точки)» и «+ Chronos-2» считаются одинаково — их и сравнивать.')
print('«Последнее значение (без сезонности)» — простейшая база: если Chronos-2 не лучше её, выигрыш у моделей отчёта объясняется не моделью, а тем, что на периоде слома выгоднее не ждать роста.')
display(fmt)

Средняя ошибка |ln(факт/прогноз)|, горизонт 1–4 кв. Модели отчёта — согласованные прогнозы (как в табл. 8);
покрытие 80 %-интервала — без калибровки ширины. «Комбинация 3 (точки)» и «+ Chronos-2» считаются одинаково — их и сравнивать.
«Последнее значение (без сезонности)» — простейшая база: если Chronos-2 не лучше её, выигрыш у моделей отчёта объясняется не моделью, а тем, что на периоде слома выгоднее не ждать роста.


                                        Рынок     ДВС Электро  \
model                                                           
Последнее значение (без сезонности)    26.3 %  41.8 %  19.9 %   
Наивный                                33.1 %  53.5 %  23.9 %   
Сезонный наивный                       41.1 %  75.6 %  23.3 %   
ETS                                    31.5 %  53.7 %  21.6 %   
ARX-ставка                             29.6 %  53.8 %  18.4 %   
Комбинация                             29.3 %  50.5 %  19.9 %   
Chronos-2                              30.0 %  54.0 %  16.4 %   
Chronos-2 + ставка                     28.3 %  47.8 %  17.4 %   
Комбинация 3 (точки) — база сравнения  30.1 %  50.0 %  19.9 %   
Комбинация 3 + Chronos-2               29.8 %  51.0 %  17.9 %   
Комбинация 3 + Chronos-2 со ставкой    29.4 %  49.3 %  18.0 %   

                                      8 крупных сегментов Доля электро, п. п.  \
model                                                                    

**Вывод по таблице.** Простая база «последнее значение» лучше моделей отчёта по рынку (26,3 % против 29,2 % у Комбинации) и ДВС (41,8 % против 50,1 %); по электро Комбинация и база равны (19,9 %), а Chronos-2 лучше (16,4 %). Chronos-2 без факторов (30,0 % / 54,0 % / 16,4 %) близок к Комбинации. Вариант со ставкой (28,3 % / 47,8 % / 17,4 %) получает фактическую будущую ставку, поэтому оптимистичен. Покрытие 80 %-ного интервала у Chronos-2 около 84 %, у моделей отчёта без калибровки 63–82 %.

**Поправка к выводу ячейки.** Строки ARX-ставка и Комбинация в выводе выше посчитаны до исправления проверки (в ней ставка после начала прогноза брала фактические значения). После исправления: ARX 29,2 % / 52,8 % / 18,3 % (было 29,6 % / 53,8 % / 18,4 %), Комбинация 29,2 % / 50,1 % / 19,9 % (было 29,3 % / 50,5 % / 19,9 %). Актуальные числа лежат в `chronos2/results/summary_table.csv`. Строки Chronos-2 не менялись.

Следующая ячейка: средняя ошибка по горизонту 1–4 квартала для рынка и ДВС. На горизонте в один квартал модели близки, дальше разброс растёт. Столбцы 3 и 4 у вариантов со ставкой оптимистичны: на этих горизонтах они используют фактическую будущую ставку (лаг 2 квартала). В строках ARX и Комбинация вывод ниже посчитан до исправления проверки, актуальные значения — в `error_by_horizon.csv`.

In [5]:
# Ошибка по горизонту: рынок и ДВС
H_T = ALL[ALL.node.isin(['Рынок', 'ДВС'])].groupby(['node', 'model', 'h']).abs_log_err.mean().unstack('h')
H_T = H_T.loc[[(n, m) for n in ['Рынок', 'ДВС'] for m in order if (n, m) in H_T.index]]
display((H_T * 100).round(1))

h                                               1     2     3     4
node  model                                                        
Рынок Последнее значение (без сезонности)    19.7  24.8  28.7  42.2
      Наивный                                18.3  34.4  40.1  57.3
      Сезонный наивный                       37.0  44.8  40.2  45.6
      ETS                                    20.8  35.2  37.5  41.8
      ARX-ставка                             22.4  33.2  33.3  34.7
      Комбинация                             18.4  32.7  34.7  41.7
      Chronos-2                              22.3  27.7  35.8  44.9
      Chronos-2 + ставка                     21.2  25.1  34.0  44.2
      Комбинация 3 (точки) — база сравнения  19.5  33.6  36.0  41.2
      Комбинация 3 + Chronos-2               19.2  32.1  35.9  42.1
      Комбинация 3 + Chronos-2 со ставкой    19.0  31.4  35.5  41.9
ДВС   Последнее значение (без сезонности)    27.0  41.7  45.7  73.3
      Наивный                                32.

## 5. Прогноз на II кв. 2026 — I кв. 2027 (от последнего факта)
Для сопоставимости с Chronos-2 берётся **сумма поквартальных медиан** модели отчёта (таблица «Прогноз_4кв»): рынок 87,4, ДВС 29,1, электро 57,5 тыс. шт.; факт последних 4 кварталов — 76,0 / 21,6 / 54,4. P10–P90 у Chronos-2 — сумма квартальных квантилей (грубая оценка, шире интервала суммы).

In [6]:
OURS = {'Рынок': 87414, 'ДВС': 29096, 'Электро': 57486}
FACT = {'Рынок': 75990, 'ДВС': 21610, 'Электро': 54380}
rows = []
for with_kr, name in [(False, 'Chronos-2'), (True, 'Chronos-2 + ставка')]:
    fc, tot = forecast_4q(predict, with_kr)
    FCS[name + ' 2026Q2–2027Q1'] = fc
    for n in ['Рынок', 'ДВС', 'Электро']:
        rows.append({'модель': name, 'ряд': n, 'медиана, тыс.': tot.loc[n, 'q0.5'] / 1e3,
                     'P10–P90 (грубо), тыс.': f"{tot.loc[n, 'q0.1']/1e3:.1f}–{tot.loc[n, 'q0.9']/1e3:.1f}",
                     'к факту посл. 4 кв.': f"{tot.loc[n, 'q0.5']/FACT[n]-1:+.0%}",
                     'модель отчёта, тыс.': OURS[n] / 1e3})
F4 = pd.DataFrame(rows); display(F4.round(1))

               модель      ряд  медиана, тыс. P10–P90 (грубо), тыс.  \
0           Chronos-2    Рынок           69.5            51.1–105.0   
1           Chronos-2      ДВС           21.3             15.9–42.6   
2           Chronos-2  Электро           50.9             32.4–75.6   
3  Chronos-2 + ставка    Рынок           73.3            51.6–113.4   
4  Chronos-2 + ставка      ДВС           23.2             15.7–46.9   
5  Chronos-2 + ставка  Электро           52.0             34.0–75.6   

  к факту посл. 4 кв.  модель отчёта, тыс.  
0                 -9%                 87.4  
1                 -1%                 29.1  
2                 -6%                 57.5  
3                 -4%                 87.4  
4                 +7%                 29.1  
5                 -4%                 57.5  

## 6. Saving results

In [7]:
import os
out = str(OUT_DIR); os.makedirs(out, exist_ok=True)
ALL.to_csv(f'{out}/backtest_all_models.csv', index=False, encoding='utf-8-sig')
NOTES = {'ARX-ставка': 'rate after the origin is frozen at the last known value (no look-ahead)',
         'Комбинация': 'rate after the origin is frozen at the last known value (no look-ahead)',
         'Комбинация 3 (точки) — база сравнения': 'rate after the origin is frozen at the last known value (no look-ahead)',
         'Chronos-2 + ставка': 'ORACLE: uses the actual future rate, optimistic',
         'Комбинация 3 + Chronos-2': 'ARX part computed with the old backtest that used the actual future rate',
         'Комбинация 3 + Chronos-2 со ставкой': 'ORACLE: actual future rate in Chronos-2 and in the ARX part, optimistic'}
T.assign(note=T.index.map(NOTES).fillna('')).to_csv(f'{out}/summary_table.csv', encoding='utf-8-sig')
H_T.to_csv(f'{out}/error_by_horizon.csv', encoding='utf-8-sig')
F4.to_csv(f'{out}/forecast_4q_chronos2.csv', index=False, encoding='utf-8-sig')
for k, v in FCS.items():
    v.to_csv(f'{out}/quantiles_{k.replace(" ", "_").replace("+", "plus")}.csv', index=False, encoding='utf-8-sig')
print('===== summary table, error by horizon, 4Q forecast =====')
print(T.round(4).to_csv(sep=';'))
print((H_T * 100).round(1).to_csv(sep=';'))
print(F4.round(1).to_csv(sep=';', index=False))
print('Results saved to:', out)


===== summary table, error by horizon, 4Q forecast =====
model;Рынок;ДВС;Электро;8 крупных сегментов;Доля электро, п. п.;Смещение рынка;Покрытие 80 %-интервала
Последнее значение (без сезонности);0.263;0.4183;0.1994;0.3893;7.1254;0.2706;
Наивный;0.3315;0.5348;0.2387;0.4175;7.4952;0.3848;0.8187
Сезонный наивный;0.4114;0.7563;0.2326;0.5184;12.2052;0.5089;0.6703
ETS;0.3152;0.5366;0.2164;0.3979;8.2317;0.3692;0.6923
ARX-ставка;0.2959;0.5378;0.1836;0.411;8.0618;0.335;0.6264
Комбинация;0.2931;0.5048;0.1995;0.3826;7.3262;0.3403;0.6978
Chronos-2;0.2999;0.5401;0.1638;0.4149;9.432;0.337;0.8407
Chronos-2 + ставка;0.2832;0.478;0.1736;0.4022;7.7937;0.3159;0.8352
Комбинация 3 (точки) — база сравнения;0.3014;0.5004;0.1993;0.3957;7.3826;0.3499;
Комбинация 3 + Chronos-2;0.2977;0.5104;0.1791;0.3948;7.7611;0.3467;
Комбинация 3 + Chronos-2 со ставкой;0.2937;0.4931;0.1801;0.3914;7.4232;0.3414;

node;model;1;2;3;4
Рынок;Последнее значение (без сезонности);19.7;24.8;28.7;42.2
Рынок;Наивный;18.3;34.4;40.1;57.3

**Вывод по сохранению.** В `chronos2/results/` записаны `summary_table.csv` (колонка `note` помечает строки с фактической будущей ставкой и пересчитанные строки), `error_by_horizon.csv` и `forecast_4q_chronos2.csv`. Квантили по каждому прогону и `backtest_all_models.csv` в репозиторий не включены, поэтому строки Chronos без повторного запуска модели не воспроизвести.

## Итоговые выводы
Основано на уже сохранённых выводах ячеек выше (ноутбук заново не запускался) и на пересчитанных строках пайплайна в `chronos2/results/summary_table.csv`.

1. **Простая база лучше моделей отчёта по рынку и ДВС.** «Последнее значение без сезонности»: рынок 26,3 %, ДВС 41,8 %, электро 19,9 %. Комбинация отчёта: 29,2 % / 50,1 % / 19,9 %. На 13 кварталах после структурного слома рынка выигрывает модель, которая просто не ждёт отскока.
2. **Chronos-2 без факторов не лучше базы и близок к Комбинации:** 30,0 % / 54,0 % / 16,4 %. Заметно лучше только по электро (16,4 % против 19,9 %); по ДВС хуже Комбинации.
3. **Вариант со ставкой нельзя считать честным выигрышем.** Chronos-2 + ставка (28,3 % / 47,8 % / 17,4 %) использует фактическую будущую ставку на горизонтах 3–4. Тот же приём в старой проверке ARX был ошибкой; после исправления ARX даёт 29,2 % / 52,8 % / 18,3 % (раньше 29,6 % / 53,8 % / 18,4 %).
4. **Прогнозы на следующие 4 квартала расходятся.** Chronos-2 даёт 69,5 тыс. (со ставкой 73,3), то есть ниже факта последних четырёх кварталов (−9 % и −4 %); модель отчёта — 87,4 тыс. (+15 %). Эта разница — ещё один довод не опираться на верхнюю половину интервала отчёта.
5. **Ограничения.** 5 начал прогноза, один слом, горизонт 1–4; проверка на покой не проводилась. Строки Chronos не пересчитывались после исправления проверки ставки: для этого нужна модель (запуск на GPU/CPU).

## Источники
1. Ansari A.F. et al. Chronos-2: From Univariate to Universal Forecasting. arXiv:2510.15821, 2025. https://arxiv.org/abs/2510.15821
2. Ansari A.F. et al. Chronos: Learning the Language of Time Series. Transactions on Machine Learning Research, 2024. arXiv:2403.07815.
3. Код и веса: https://github.com/amazon-science/chronos-forecasting ; модель `amazon/chronos-2` (Hugging Face).
4. Данные и бэктест моделей отчёта: `data/processed/market_long.csv`, `macro_quarterly_wide.csv`, `backtest_rolling_origin.csv` этого репозитория (ноутбуки 01–04).